# Tarea downstream con Keras: clasificación sobre imágenes fusionadas

Verifica la **utilidad** de la fusión en un segundo tipo de modelo (independiente de YOLO): un clasificador por **transfer learning (MobileNetV2)** que decide, por imagen, una etiqueta de interés (p. ej. *persona presente* vs *ausente*). Compara métricas entre VIS, IR y las fusiones, y reporta **accuracy, precision, recall, F1**, matriz de confusión y curvas; mide producción (latencia, tamaño, TFLite). Ejecutar en **Colab (GPU)**.


## 1. Instalación e imports


In [ ]:
!pip -q install tensorflow scikit-learn
import tensorflow as tf, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## 2. Datos
Organiza las imágenes (de cada versión fusionada) en carpetas por clase:
`dataset/<train|val>/<clase>/*.png`. Las etiquetas a nivel imagen pueden derivarse de las anotaciones de Roboflow (imagen con/sin objeto) o definirse según tu problema.


In [ ]:
IMG=224; BATCH=16; VARIANT='Fused_WTH_diskL5'  # cambia para comparar versiones
train = tf.keras.utils.image_dataset_from_directory(f'dataset_{VARIANT}/train', image_size=(IMG,IMG), batch_size=BATCH)
val   = tf.keras.utils.image_dataset_from_directory(f'dataset_{VARIANT}/val',   image_size=(IMG,IMG), batch_size=BATCH)
classes = train.class_names; print(classes)
norm = tf.keras.layers.Rescaling(1./255)
train = train.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)
val   = val.map(lambda x,y:(norm(x),y)).prefetch(tf.data.AUTOTUNE)


## 3. Modelo (transfer learning)


In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(IMG,IMG,3), include_top=False, weights='imagenet')
base.trainable = False
model = tf.keras.Sequential([
    base, tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(classes), activation='softmax')])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')])
hist = model.fit(train, validation_data=val, epochs=15)


## 4. Métricas y curvas


In [ ]:
import numpy as np
y_true=np.concatenate([y for _,y in val]); y_prob=model.predict(val); y_pred=y_prob.argmax(1)
print(classification_report(y_true, y_pred, target_names=classes, digits=4))
print('Matriz de confusión:\n', confusion_matrix(y_true, y_pred))
plt.plot(hist.history['accuracy'],label='train'); plt.plot(hist.history['val_accuracy'],label='val')
plt.title('Accuracy'); plt.legend(); plt.show()


## 5. Producción: latencia, tamaño y TFLite


In [ ]:
import time, os
x=np.random.rand(1,IMG,IMG,3).astype('float32')
t=time.time();
for _ in range(50): model.predict(x, verbose=0)
print(f'latencia ~ {(time.time()-t)/50*1000:.1f} ms/img')
conv=tf.lite.TFLiteConverter.from_keras_model(model); tfl=conv.convert()
open('modelo.tflite','wb').write(tfl)
print('TFLite MB:', os.path.getsize('modelo.tflite')/1e6)


## 6. Comparación entre versiones
Repite el entrenamiento cambiando `VARIANT` (VIS, IR, Fused_WTH_diskL5, Fused_Laplace, …), guarda accuracy/precision/recall/F1 de cada una en un DataFrame y aplica Friedman/Wilcoxon (con corrección de Holm) para comparar formalmente, como en el resto de la tesis.
